# PW-FNet deraining block for fog-adaptive ANPR (Kaggle, T4 GPU)

Re-implementation of PW-FNet ("Global Modeling Matters", arXiv 2507.13663), written from the paper since the official repo's data/weights are Baidu-only.

**Run order:** 0 (GPU check) → 1 (model) → 2 (utils) → 3 (sanity) → 4 (get Rain13K) → 5 (get plate data) → 6 (build datasets) → 7 (train stage 1) → 8 (train stage 2) → 9 (evaluate) → 10 (export).

**IMPORTANT — Kaggle sessions can reset and wipe /kaggle/working.** After every checkpoint save, download it immediately (hover the file in the Output panel → ⋮ → Download). Once Rain13K is uploaded as your own Kaggle dataset, cell 4 below is skippable — just attach the dataset instead.

## 0. GPU check

In [ ]:
!nvidia-smi | head -15
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

## 1. Model (PW-MIMO + Fourier blocks)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# ---------- Haar wavelet (lossless, exactly invertible) ----------
def dwt(x):
    a, b = x[..., 0::2, 0::2], x[..., 0::2, 1::2]
    c, d = x[..., 1::2, 0::2], x[..., 1::2, 1::2]
    ll = (a + b + c + d) / 2
    lh = (a + b - c - d) / 2
    hl = (a - b + c - d) / 2
    hh = (a - b - c + d) / 2
    return torch.cat([ll, lh, hl, hh], 1)          # (B,4C,H/2,W/2)

def idwt(x):
    C = x.shape[1] // 4
    ll, lh, hl, hh = x[:, :C], x[:, C:2*C], x[:, 2*C:3*C], x[:, 3*C:]
    a = (ll + lh + hl + hh) / 2
    b = (ll + lh - hl - hh) / 2
    c = (ll - lh + hl - hh) / 2
    d = (ll - lh - hl + hh) / 2
    B, _, H, W = ll.shape
    out = x.new_zeros(B, C, H * 2, W * 2)
    out[..., 0::2, 0::2], out[..., 0::2, 1::2] = a, b
    out[..., 1::2, 0::2], out[..., 1::2, 1::2] = c, d
    return out

class LayerNorm2d(nn.Module):
    def __init__(s, c, eps=1e-6):
        super().__init__(); s.w = nn.Parameter(torch.ones(c)); s.b = nn.Parameter(torch.zeros(c)); s.eps = eps
    def forward(s, x):
        mu = x.mean(1, keepdim=True); var = (x - mu).pow(2).mean(1, keepdim=True)
        return (x - mu) / torch.sqrt(var + s.eps) * s.w[None, :, None, None] + s.b[None, :, None, None]

# ---------- PW-FNet block: Fourier token mixer + Uformer-style FFN ----------
class FourierMixer(nn.Module):
    """1x1 (C->2C) -> FFT -> 1x1 + GELU (on real/imag) -> IFFT -> 1x1 (2C->C)"""
    def __init__(s, c):
        super().__init__()
        s.up = nn.Conv2d(c, 2 * c, 1)
        s.freq = nn.Conv2d(4 * c, 4 * c, 1)      # real+imag stacked -> 2C*2 channels
        s.down = nn.Conv2d(2 * c, c, 1)
    def forward(s, x):
        x = s.up(x)
        H, W = x.shape[-2:]
        with torch.autocast(device_type="cuda", enabled=False):   # FFT in fp32 (fp16 FFT needs pow-2 sizes)
            f = torch.fft.rfft2(x.float(), norm="backward")
            f = torch.cat([f.real, f.imag], 1)
            f = F.gelu(s.freq(f))
            r, i = f.chunk(2, 1)
            x = torch.fft.irfft2(torch.complex(r, i), s=(H, W), norm="backward")
        return s.down(x)

class FFN(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.a = nn.Conv2d(c, 2 * c, 1); s.dw = nn.Conv2d(2 * c, 2 * c, 3, padding=1, groups=2 * c); s.b = nn.Conv2d(2 * c, c, 1)
    def forward(s, x): return s.b(F.gelu(s.dw(s.a(x))))

class Block(nn.Module):
    def __init__(s, c):
        super().__init__()
        s.n1, s.mix, s.n2, s.ffn = LayerNorm2d(c), FourierMixer(c), LayerNorm2d(c), FFN(c)
    def forward(s, x):
        x = x + s.mix(s.n1(x))
        return x + s.ffn(s.n2(x))

def stage(c, n): return nn.Sequential(*[Block(c) for _ in range(n)])

# ---------- PW-MIMO network ----------
class PWFNet(nn.Module):
    def __init__(s, width=32, enc=(2, 2, 4), dec=(2, 2, 2), mid=2):
        super().__init__()
        c1, c2, c3 = width, width * 2, width * 4
        # multi-input: I^1 (3ch), I^2 = DWT(I) (12ch), I^4 = DWT(DWT(I)) (48ch)
        s.in1, s.in2, s.in4 = nn.Conv2d(3, c1, 3, padding=1), nn.Conv2d(12, c2, 3, padding=1), nn.Conv2d(48, c3, 3, padding=1)
        s.fuse2, s.fuse3 = nn.Conv2d(c2 * 2, c2, 1), nn.Conv2d(c3 * 2, c3, 1)
        s.e1, s.e2, s.e3 = stage(c1, enc[0]), stage(c2, enc[1]), stage(c3, enc[2])
        s.d12, s.d23 = nn.Conv2d(4 * c1, c2, 1), nn.Conv2d(4 * c2, c3, 1)     # trainable wavelet downsampling (DWT -> 1x1)
        s.mid = stage(c3, mid)
        s.u32, s.u21 = nn.Conv2d(c3, 4 * c2, 1), nn.Conv2d(c2, 4 * c1, 1)      # trainable inverse-wavelet upsampling (1x1 -> IWT)
        s.f2, s.f1 = nn.Conv2d(c2 * 2, c2, 1), nn.Conv2d(c1 * 2, c1, 1)         # skip fusion
        s.dd3, s.dd2, s.dd1 = stage(c3, dec[2]), stage(c2, dec[1]), stage(c1, dec[0])
        s.o4, s.o2, s.o1 = nn.Conv2d(c3, 48, 3, padding=1), nn.Conv2d(c2, 12, 3, padding=1), nn.Conv2d(c1, 3, 3, padding=1)
        for m in (s.o4, s.o2, s.o1):            # zero-init residual heads -> network starts as identity (out = input)
            nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)

    def forward(s, x, stage_out="all"):
        H, W = x.shape[-2:]
        assert H % 4 == 0 and W % 4 == 0, "pad input to a multiple of 4"
        i1 = x; i2 = dwt(i1); i4 = dwt(i2)
        f1 = s.e1(s.in1(i1))
        f2 = s.e2(s.fuse2(torch.cat([s.d12(dwt(f1)), s.in2(i2)], 1)))
        f3 = s.e3(s.fuse3(torch.cat([s.d23(dwt(f2)), s.in4(i4)], 1)))
        z = s.mid(f3)
        z = s.dd3(z)
        out4 = idwt(idwt(s.o4(z) + i4))                                 # small  : image at full res, from 1/4-scale branch
        if stage_out == "small": return out4
        z = idwt(s.u32(z)); z = s.dd2(s.f2(torch.cat([z, f2], 1)))
        out2 = idwt(s.o2(z) + i2)                                  # medium
        if stage_out == "medium": return out2
        z = idwt(s.u21(z)); z = s.dd1(s.f1(torch.cat([z, f1], 1)))
        out1 = s.o1(z) + i1                                        # large
        if stage_out == "large": return out1
        return out1, out2, out4

def fourier_l1(pred, gt):
    return (torch.fft.fft2(pred.float()) - torch.fft.fft2(gt.float())).abs().mean()

def pwfnet_loss(outs, gt):
    return sum(fourier_l1(o, gt) for o in outs)

## 2. Training / data / eval utilities

In [ ]:
import os, re, random, time, math, zipfile
from pathlib import Path
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from PIL import Image
from skimage.metrics import structural_similarity

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp"}

# ---------------- pairing / splitting ----------------
def list_imgs(d):
    return sorted(p for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXT)

def default_key(p):
    """rain-001 / norain-001 / 001x2 / 001_rain  ->  '001'"""
    s = p.stem.lower()
    for tok in ("norain", "rain", "x2", "_clean", "_gt", "_target", "_input"):
        s = s.replace(tok, "")
    return s.strip("-_ ")

def pair_by_dirs(rain_dir, clean_dir, key=default_key):
    clean = {key(p): p for p in list_imgs(clean_dir)}
    pairs = [(p, clean[key(p)]) for p in list_imgs(rain_dir) if key(p) in clean]
    print(f"{len(pairs)} pairs  ({rain_dir} <-> {clean_dir})")
    for a, b in pairs[:3]: print("   ", a.name, "<->", b.name)
    return pairs

def group_split(pairs, plate_id_fn, val_frac=0.1, test_frac=0.1, seed=0):
    """Split by SOURCE PLATE so the same plate never appears in train and test (prevents leakage)."""
    ids = sorted({plate_id_fn(a) for a, _ in pairs})
    random.Random(seed).shuffle(ids)
    n_te, n_va = int(len(ids) * test_frac), int(len(ids) * val_frac)
    te, va = set(ids[:n_te]), set(ids[n_te:n_te + n_va])
    out = {"train": [], "val": [], "test": []}
    for a, b in pairs:
        pid = plate_id_fn(a)
        out["test" if pid in te else "val" if pid in va else "train"].append((a, b))
    print({k: len(v) for k, v in out.items()}, "| plates:", len(ids))
    return out

# ---------------- dataset ----------------
def load_rgb(p): return np.array(Image.open(p).convert("RGB"))

class PairDS(Dataset):
    def __init__(self, pairs, patch=128, train=True, cache=True):
        self.pairs, self.patch, self.train = pairs, patch, train
        self.cache = [(load_rgb(a), load_rgb(b)) for a, b in pairs] if cache else None
    def __len__(self): return len(self.pairs)
    def _get(self, i):
        return self.cache[i] if self.cache else (load_rgb(self.pairs[i][0]), load_rgb(self.pairs[i][1]))
    def __getitem__(self, i):
        x, y = self._get(i)
        if y.shape != x.shape:                      # safety: force same size
            y = np.asarray(Image.fromarray(y).resize((x.shape[1], x.shape[0]), Image.BICUBIC))
        if self.train:
            P = self.patch
            h, w = x.shape[:2]
            if h < P or w < P:                      # reflect-pad small plate crops
                ph, pw = max(0, P - h), max(0, P - w)
                x = np.pad(x, ((0, ph), (0, pw), (0, 0)), mode="reflect"); y = np.pad(y, ((0, ph), (0, pw), (0, 0)), mode="reflect")
                h, w = x.shape[:2]
            t, l = random.randint(0, h - P), random.randint(0, w - P)
            x, y = x[t:t+P, l:l+P], y[t:t+P, l:l+P]
            if random.random() < .5: x, y = x[:, ::-1], y[:, ::-1]
            if random.random() < .5: x, y = x[::-1], y[::-1]
            k = random.randint(0, 3)
            x, y = np.rot90(x, k), np.rot90(y, k)
        to_t = lambda a: torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float() / 255.
        return to_t(x), to_t(y)

# ---------------- inference helpers ----------------
def pad4(x):
    H, W = x.shape[-2:]
    ph, pw = (4 - H % 4) % 4, (4 - W % 4) % 4
    if ph or pw: x = torch.nn.functional.pad(x, (0, pw, 0, ph), mode="reflect")
    return x, H, W

@torch.no_grad()
def restore(model, x, stage="large"):
    """x: (B,3,H,W) in [0,1] -> restored, same size."""
    model.eval()
    xp, H, W = pad4(x.to(DEVICE))
    return model(xp, stage)[..., :H, :W].clamp(0, 1).cpu()

def psnr(a, b): return float(10 * torch.log10(1 / torch.mean((a - b) ** 2).clamp_min(1e-10)))
def ssim(a, b):
    a, b = a[0].permute(1, 2, 0).numpy(), b[0].permute(1, 2, 0).numpy()
    return structural_similarity(a, b, channel_axis=2, data_range=1.0)

@torch.no_grad()
def evaluate(model, ds, stage="large", max_n=None):
    ps, ss = [], []
    for i in range(min(len(ds), max_n or len(ds))):
        x, y = ds[i]; x, y = x[None], y[None]
        out = restore(model, x, stage)
        ps.append(psnr(out, y)); ss.append(ssim(out, y))
    return float(np.mean(ps)), float(np.mean(ss))

# ---------------- training ----------------
def train(model, train_ds, val_ds, iters, lr, bs, ckpt, loss_fn, val_every=2000, warmup=200, num_workers=2, amp=True):
    model.to(DEVICE)
    dl = DataLoader(train_ds, bs, shuffle=True, num_workers=num_workers, drop_last=True, pin_memory=True, persistent_workers=num_workers > 0)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda i: min(1, (i + 1) / warmup) * (1e-6 / lr + (1 - 1e-6 / lr) * 0.5 * (1 + math.cos(math.pi * min(i, iters) / iters))))
    use_amp = amp and DEVICE == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    it, best = 0, -1
    if os.path.exists(ckpt):                                    # resume after a Kaggle timeout
        c = torch.load(ckpt, map_location=DEVICE)
        model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"]); sched.load_state_dict(c["sched"]); scaler.load_state_dict(c["scaler"])
        it, best = c["it"], c["best"]; print(f"resumed {ckpt} at iter {it}")
    t0, run = time.time(), 0.
    while it < iters:
        for x, y in dl:
            model.train()
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                outs = model(x)
            loss = loss_fn(outs, y)                               # Fourier loss is computed in fp32
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            it += 1; run = 0.98 * run + 0.02 * loss.item() if run else loss.item()
            if it % 100 == 0: print(f"it {it}/{iters}  loss {run:.4f}  lr {sched.get_last_lr()[0]:.2e}  {time.time()-t0:.0f}s")
            if it % val_every == 0 or it == iters:
                p, s = evaluate(model, val_ds, "large", max_n=100)
                if p > best:
                    best = p; torch.save(model.state_dict(), ckpt.replace(".pt", "_best.pt"))
                print(f"  [val] PSNR {p:.2f}  SSIM {s:.4f}  (best {best:.2f})")
                torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(), it=it, best=best), ckpt)
            if it >= iters: break
    return best

## 3. Sanity tests (must all pass before training)

In [ ]:
x = torch.rand(2, 3, 64, 96)
assert torch.allclose(idwt(dwt(x)), x, atol=1e-5), "wavelet is not lossless"
m = PWFNet(24)
print("params (M):", round(sum(p.numel() for p in m.parameters()) / 1e6, 3))
outs = m(x); print([o.shape for o in outs])
assert all(torch.allclose(o, x, atol=1e-5) for o in outs), "should start as identity (zero-init heads)"
print("stage inference OK:", [m(x, s).shape for s in ("small", "medium", "large")])
print(m(torch.rand(1, 3, 100, 180))[0].shape, "<- odd sizes OK")
del m

## 4. Get Rain13K (stage-1 pretrain data)

**Option A — if you've uploaded `rain13k-train-data` as your own Kaggle dataset (recommended, instant, no rate limits):**
Add Input → Your Datasets → attach it, then just point the paths at it (edit the folder name below to match).

**Option B — download fresh from Hugging Face (slow, can hit rate limits — needs an HF_TOKEN secret):**

In [ ]:
# ---- Option A: already uploaded as a Kaggle dataset ----
PRETRAIN_RAIN_DIR  = "/kaggle/input/rain13k-train-data/input"   # <- fix folder name to match your dataset
PRETRAIN_CLEAN_DIR = "/kaggle/input/rain13k-train-data/target"

# ---- Option B: fresh download (uncomment if Option A's path doesn't exist) ----
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import snapshot_download
# tok = UserSecretsClient().get_secret("HF_TOKEN")
# snapshot_download("dronefreak/Rain13K", repo_type="dataset", allow_patterns=["train/*"],
#                    local_dir="/kaggle/working/rain13k", token=tok, max_workers=4)
# PRETRAIN_RAIN_DIR  = "/kaggle/working/rain13k/train/input"
# PRETRAIN_CLEAN_DIR = "/kaggle/working/rain13k/train/target"

from pathlib import Path
for d, p in [("input", PRETRAIN_RAIN_DIR), ("target", PRETRAIN_CLEAN_DIR)]:
    print(d, len(list(Path(p).rglob("*.jpg"))), "(expect 13711)")

## 5. Plate rain/clean pairs (stage-2 fine-tune data)

In [ ]:
import re
PLATE_RAIN_DIR  = "/kaggle/input/datasets/meghats04/restoration-train-zip/restoration_train/degraded"
PLATE_CLEAN_DIR = "/kaggle/input/datasets/meghats04/restoration-train-zip/restoration_train/clean"

plate_key = lambda p: p.stem
_deg_re = re.compile(r"_(fog|rain)_(light|moderate|heavy|severe)$")
plate_id = lambda p: _deg_re.sub("", p.stem)
is_rain  = lambda p: "_rain_" in p.name.lower()

## 6. Pair, split (by plate — no leakage), build datasets

In [ ]:
pre_pairs = pair_by_dirs(PRETRAIN_RAIN_DIR, PRETRAIN_CLEAN_DIR)
plate_pairs = [p for p in pair_by_dirs(PLATE_RAIN_DIR, PLATE_CLEAN_DIR, key=plate_key) if is_rain(p[0])]
split = group_split(plate_pairs, lambda p: plate_id(p), val_frac=0.1, test_frac=0.1)

PATCH = 128
ds_pre_tr, ds_pre_va = PairDS(pre_pairs[:-100], PATCH, True), PairDS(pre_pairs[-100:], PATCH, False)
ds_pl_tr = PairDS(split["train"], PATCH, True)
ds_pl_va = PairDS(split["val"], PATCH, False)
ds_pl_te = PairDS(split["test"], PATCH, False)          # touch ONLY at the end

print("pretrain:", len(ds_pre_tr), len(ds_pre_va))
print("plates:  ", len(ds_pl_tr), len(ds_pl_va), len(ds_pl_te))

## 7. Stage 1 — pretrain on Rain13K
~36s/100 iters on a T4. **Download `pwfnet_pre_best.pt` the moment a validation prints a new best — don't wait for the run to finish.**

In [ ]:
model = PWFNet(width=24, enc=(2, 2, 4), dec=(2, 2, 2), mid=2)
ITERS_PRE = 40000       # lower to ~15000-20000 first if you want a faster, safer first checkpoint
best = train(model, ds_pre_tr, ds_pre_va, iters=ITERS_PRE, lr=5e-4, bs=16,
             ckpt="/kaggle/working/pwfnet_pre.pt", loss_fn=pwfnet_loss, val_every=2000)

## 8. Stage 2 — fine-tune on plate rain pairs

In [ ]:
model = PWFNet(width=24, enc=(2, 2, 4), dec=(2, 2, 2), mid=2)
model.load_state_dict(torch.load("/kaggle/working/pwfnet_pre_best.pt", map_location=DEVICE))

ITERS_FT = 15000
best = train(model, ds_pl_tr, ds_pl_va, iters=ITERS_FT, lr=2e-4, bs=16,
             ckpt="/kaggle/working/pwfnet_ft.pt", loss_fn=pwfnet_loss, val_every=1000)

## 9. Evaluate on held-out plates

In [ ]:
model.load_state_dict(torch.load("/kaggle/working/pwfnet_ft_best.pt", map_location=DEVICE)); model.to(DEVICE).eval()

ps = [psnr(ds_pl_te[i][0][None], ds_pl_te[i][1][None]) for i in range(len(ds_pl_te))]
ss = [ssim(ds_pl_te[i][0][None], ds_pl_te[i][1][None]) for i in range(len(ds_pl_te))]
print(f"rainy input   PSNR {np.mean(ps):.2f}  SSIM {np.mean(ss):.4f}")
for st in ("small", "medium", "large"):
    p, s = evaluate(model, ds_pl_te, st); print(f"PW-FNet-{st:<6} PSNR {p:.2f}  SSIM {s:.4f}")

In [ ]:
import matplotlib.pyplot as plt
idx = np.linspace(0, len(ds_pl_te) - 1, 6).astype(int)
fig, ax = plt.subplots(len(idx), 3, figsize=(9, 2.2 * len(idx)))
for r, i in enumerate(idx):
    x, y = ds_pl_te[i]; o = restore(model, x[None])[0]
    for c, im in enumerate((x, o, y)):
        ax[r, c].imshow(im.permute(1, 2, 0).numpy()); ax[r, c].axis("off")
for c, t in enumerate(("rainy", "PW-FNet", "clean")): ax[0, c].set_title(t)
plt.tight_layout(); plt.show()

## 10. OCR check + export

In [ ]:
def ocr_eval(items, restore_fn, ocr_fn):
    """items: list of (rainy_image_path, true_plate_text)."""
    exact, chars, n = 0, 0, 0
    for path, gt in items:
        img = load_rgb(path)
        if restore_fn is not None:
            t = torch.from_numpy(img).permute(2, 0, 1).float()[None] / 255.
            img = (restore_fn(t)[0].permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)
        pred = ocr_fn(img).strip().upper().replace(" ", "")
        gt = gt.upper().replace(" ", "")
        exact += pred == gt; chars += sum(a == b for a, b in zip(pred, gt)) / max(len(gt), 1); n += 1
    return exact / n, chars / n

# items = [(path, "KL07AB1234"), ...]; ocr_fn = your_ocr_function
# print("none   ", ocr_eval(items, None, ocr_fn))
# print("PW-FNet", ocr_eval(items, lambda t: restore(model, t), ocr_fn))

In [ ]:
torch.save({"state_dict": model.state_dict(), "cfg": dict(width=24, enc=(2, 2, 4), dec=(2, 2, 2), mid=2)},
           "/kaggle/working/pwfnet_derain_final.pt")

def derain(img_rgb_uint8, stage="large"):
    t = torch.from_numpy(img_rgb_uint8).permute(2, 0, 1).float()[None] / 255.
    return (restore(model, t, stage)[0].permute(1, 2, 0).numpy() * 255).round().astype(np.uint8)